# Explore canonical SSP585 hazard curves

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RiskThinking/crc-docs/blob/main/notebooks/crc_open_hazards.ipynb) · [GitHub preview](https://github.com/RiskThinking/crc-docs/blob/main/notebooks/crc_open_hazards.ipynb)

This notebook pairs with the [CRC portfolio pipeline](../pipelines/crc_open_pipeline.py). Start with the catalogue, evaluate Rx1day (annual maximum one-day precipitation) at eight sampled city locations, explore its return-value curves and horizon labels, then optionally compare a canonical ERA5 historical baseline. The Toronto 2050 results at 10 and 100 years reproduce the pipeline's example.

The [fixtures](../fixtures/crc_open/README.md) contain 61 hazards, only literal SSP585 rows and small geographic samples. They are suitable for learning the canonical workflow; they do not supply global coverage or a historical baseline. H3 cell values are not property-scale measurements.

## Prepare the runtime

Requires Python 3.12+ and published **crc-sdk 0.8.0a3 or newer**. In Colab, choose **Runtime → Run all**. Setup installs into the current kernel; restart if an already imported package changes. A crc-docs checkout uses its checked-in fixtures. Standalone runs use HTTP. No cloud credentials are needed.

In [ ]:
#@title Install dependencies and prepare this notebook
import hashlib
import importlib
import importlib.metadata
import importlib.util
import os
from pathlib import Path
import subprocess
import sys
import sysconfig
import urllib.request

if sys.version_info < (3, 12):
    raise RuntimeError("Use a Python 3.12+ runtime (Colab: Runtime > Change runtime type).")

# Install into this kernel's Python, not a separate shell environment.
REQUIREMENTS = ['crc-sdk[geometry]>=0.8.0a3,<1', 'crc-framework>=0.3.0,<1', 'duckdb>=1.4.5,<2', 'numpy>=1.26,<3', 'pandas>=2.2,<4', 'plotly==6.9.0', 'nbformat>=5.10,<6']
loaded_versions = {
    module: getattr(sys.modules[module], "__version__", None)
    for module in ("numpy", "pandas", "pyarrow", "plotly", "duckdb", "crc_sdk", "crc_framework")
    if module in sys.modules
}
if importlib.util.find_spec("pip") is None:
    subprocess.check_call([sys.executable, "-m", "ensurepip", "--upgrade"])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
    *REQUIREMENTS,
])
importlib.invalidate_caches()
for module, loaded in loaded_versions.items():
    installed = importlib.metadata.version(module.replace("_", "-"))
    if loaded is not None and loaded != installed:
        raise RuntimeError("Packages changed after import. Restart the kernel/session, then run all cells again.")
os.environ["PATH"] = sysconfig.get_path("scripts") + os.pathsep + os.environ.get("PATH", "")
IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False

REPO_ROOT = next((
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "pyproject.toml").is_file()
    and 'name = "crc-docs"' in (folder / "pyproject.toml").read_text()
), None)
print("Dependencies ready. Run the remaining cells in order.")


In [2]:
import json
from pathlib import Path

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display

from crc_sdk.providers.crc_open import CRCOpenHazards, DOCS_FIXTURES
from crc_sdk.workflows import HazardDataset

STATIC_PREVIEW = os.environ.get("CRC_NOTEBOOK_STATIC_PREVIEW") == "1"
pio.renderers.default = (
    "colab" if IN_COLAB else "plotly_mimetype+png" if STATIC_PREVIEW else "plotly_mimetype"
)


## Choose a reproducible source

`FIXTURE_ROOT` is a directory or HTTP root containing release directories. For a remote run you can replace `main` in the URL with an immutable crc-docs commit SHA. Release IDs and checksums pin the catalogue and partition bytes. Persistent `reuse` caching verifies the downloaded data; it also avoids repeated downloads on later runs.

Keep Rx1day for the comparisons below: other hazards have different units, tails and interpretation. Change `FOCUS_CITY` to explore another sampled city. An ERA5 file is optional; the [ERA5 baseline notebook](era5_historical_baseline.ipynb) produces a European baseline suitable for London.

In [3]:
RELEASE = "ssp585-fixture-2026-10-08-v2"
FIXTURE_ROOT = (
    REPO_ROOT / "fixtures/crc_open"
    if REPO_ROOT is not None and (REPO_ROOT / "fixtures/crc_open" / RELEASE).is_dir()
    else DOCS_FIXTURES
)
FOCUS_CITY = "Toronto"  # Use London with the European ERA5 baseline.
HORIZON = 2050
RETURN_PERIODS = [2, 5, 10, 20, 50, 100]
ERA5_RX1DAY = None  # Set to Path("data/era5_rx1day_europe.parquet") if available.
OUTPUT = (REPO_ROOT / "notebooks" if REPO_ROOT else Path.cwd()) / "data/crc_open"
OUTPUT.mkdir(parents=True, exist_ok=True)

provider = CRCOpenHazards(FIXTURE_ROOT, RELEASE)
catalog = provider.parse_catalog(provider.catalog_bytes())
print(f"Release: {catalog.release}; pathways: {', '.join(catalog.pathways)}")
print(f"Attribution: {catalog.attribution}; data licence: {catalog.licence}")


Release: ssp585-fixture-2026-10-08-v2; pathways: ssp585
Attribution: Riskthinking.AI bias-corrected climate distributions; data licence: pending


## Inspect coverage before calculating

Each hazard declares its native resolution, units, actual cells and available horizon labels. Rows include source no-data curves. Counts below describe the **sample**, not the underlying global dataset. `cflood` and `rflood` are excluded because their source labels are Hot House/NDC/RT3 rather than SSP585; no scenarios were relabelled.

Cyclone broadcasts time/scenario-independent curves under multiple labels, including 2010. Those labels are neither SSP585-specific projections nor a historical baseline. Inundation's probability semantics remain unspecified. Their catalogue notes are retained below.

In [4]:
coverage = pd.DataFrame([
    dict(hazard=name, unit=hazard.unit, h3_resolution=hazard.h3_resolution,
         cells=len({cell for part in hazard.partitions for cell in (part.cells or ())}),
         horizons=len(hazard.horizons), first=min(hazard.horizons), last=max(hazard.horizons),
         rows=sum(part.rows for part in hazard.partitions), notes=hazard.notes or "")
    for name, hazard in sorted(catalog.hazards.items())
])
display(coverage)
display(pd.DataFrame(catalog.excluded_hazards.items(), columns=["excluded_hazard", "reason"]))
for name, hazard in catalog.hazards.items():
    if hazard.notes:
        print(f"{name}: {hazard.notes}")


,hazard,unit,h3_resolution,cells,horizons,first,last,rows,notes
0,at,K,5,55,14,2025,2090,770,
1,bui,dimensionless,5,55,14,2025,2090,770,
2,calm_days,(m/s) days,5,55,14,2025,2090,770,
3,cold_spell_days,days,5,55,14,2025,2090,770,
4,cold_spell_frequency,events,5,55,14,2025,2090,770,
...,...,...,...,...,...,...,...,...,...
56,wbgt,K,5,55,14,2025,2090,770,
57,wet_days,days,5,56,14,2025,2090,784,
58,wind_max_daily_max,m/s,5,55,14,2025,2090,770,
59,wind_max_daily_mean,m/s,5,55,14,2025,2090,770,


,excluded_hazard,reason
0,cflood,No SSP585 pathway in inspected canonical parti...
1,rflood,No SSP585 pathway in inspected canonical parti...


cyclone: Time- and scenario-independent pooled distribution broadcast upstream across labels; SSP585-labelled rows are not SSP585 projections
inundation: Source probability semantics are unspecified; retain source values without claiming annual exceedance probabilities. Sparse inundation sampling uses up to seven available native cells within 50 km of each requested city when exact cells are absent


## Acquire canonical curves and match city assets

The SDK ingests existing canonical curves without refitting. This example selects Rx1day across the published horizons, then evaluates the catalogue's eight city coordinates. Spatial matches identify the cell used for each location. Missing matches or null values must remain missing, rather than becoming zero risk.

Climate-index curves have `annual_value_distribution` semantics and pooled ensembles. Here a return-period convention maps an upper-tail value to non-exceedance probability `1 − 1/T`. It describes a quantile of the pooled annual indicator distribution; it is not a validated annual exceedance rate for an individual asset. Pooled curves do not expose per-model ensemble spread. The source does not specify temporal window lengths.

In [5]:
assets = pa.Table.from_pylist([
    dict(asset_id=site["name"], longitude=site["longitude"], latitude=site["latitude"])
    for site in catalog.sites
])
if FOCUS_CITY not in assets.column("asset_id").to_pylist():
    raise ValueError("FOCUS_CITY must be one of the catalogue's sampled cities")
rx1day = catalog.hazards["rx1day"]
if HORIZON not in rx1day.horizons:
    raise ValueError("HORIZON must be published for Rx1day")
plan = (
    HazardDataset.crc_open(release=RELEASE, fixtures=FIXTURE_ROOT)
    .hazards(["rx1day"])
    .horizons(list(rx1day.horizons))
    .cache(OUTPUT / "cache", mode="reuse")
)
dataset = plan.materialize(OUTPUT / "rx1day.parquet")
metadata = dataset.metadata()
display(pd.DataFrame([dict(
    hazard="rx1day", unit=rx1day.unit, probability_semantics=rx1day.probability_semantics,
    pooling=rx1day.pooling, h3_resolution=rx1day.h3_resolution,
    source_provider=metadata.source.provider, release=catalog.release,
)]))
canonical = pq.read_table(OUTPUT / "rx1day.parquet")
display(canonical.select(["curve_kind"]).to_pandas().value_counts().rename("rows").to_frame())
result = (
    dataset.for_assets(assets).return_periods(RETURN_PERIODS)
    .write_parquet(OUTPUT / "city_return_values.parquet")
)
values = pq.read_table(result.output).to_pandas()
selected = values.loc[values.horizon.eq(HORIZON)].copy()
display(selected[["asset_id", "horizon", "pathway", "spatial_match", "value_rp10", "value_rp100"]])
print("Canonical curves and city return values written under data/crc_open.")


CRC fixtures provide sampled SSP585 coverage with pooled ensembles and no historical baseline. Check the catalogue for available cells and hazard-specific interpretation notes. Use source= to select another catalogue root.
  with self._prepare(progress) as (catalog, paths, hits, misses, checksum):


,hazard,unit,probability_semantics,pooling,h3_resolution,source_provider,release
0,rx1day,mm/day,annual_value_distribution,pooled,5,crc_open,ssp585-fixture-2026-10-08-v2


,rows
curve_kind,
hurdle,600
fitted,184


,asset_id,horizon,pathway,spatial_match,value_rp10,value_rp100
13,Toronto,2050,ssp585,h3_cell,58.758843,85.432655
24,Sydney,2050,ssp585,h3_cell,98.750220,169.379600
57,Phoenix,2050,ssp585,h3_cell,58.760521,99.506442
76,Nairobi,2050,ssp585,h3_cell,61.713236,109.757538
79,Singapore,2050,ssp585,h3_cell,99.496177,162.229076
83,Sao Paulo,2050,ssp585,h3_cell,79.698814,115.563877
96,Dhaka,2050,ssp585,h3_cell,263.708883,484.413493
102,London,2050,ssp585,h3_cell,42.568571,62.197652


Canonical curves and city return values written under data/crc_open.


## Compare sampled cities at one horizon

The curve joins evaluated quantiles at the chosen return periods; it is not a new fit. Click legend entries to isolate cities. Longer return periods probe the fitted tail and deserve more scrutiny. Differences describe these sampled cell distributions, not a ranking of whole cities' climate risk.

In [6]:
fig = go.Figure()
for row in selected.itertuples():
    fig.add_trace(go.Scatter(
        x=RETURN_PERIODS, y=[getattr(row, f"value_rp{period}") for period in RETURN_PERIODS],
        mode="lines+markers", name=row.asset_id, connectgaps=False,
    ))
fig.update_layout(title=f"Rx1day pooled quantiles · SSP585 · {HORIZON}",
                  xaxis=dict(title="Return-period convention (years)", type="log"),
                  yaxis_title=f"Rx1day ({rx1day.unit})", template="plotly_white")
fig.show()


## Explore the available horizons

These are source horizon labels, with unspecified window lengths. The lines connect published values without interpolation or an uncertainty band. Compare the two tail quantiles at the focus city; their separation is not ensemble uncertainty. No historical point is implied.

In [7]:
focus = values.loc[values.asset_id.eq(FOCUS_CITY)].sort_values("horizon")
if focus.empty:
    raise LookupError(f"No fixture match for {FOCUS_CITY}")
fig = go.Figure()
for period in (10, 100):
    fig.add_trace(go.Scatter(x=focus.horizon, y=focus[f"value_rp{period}"],
                            mode="lines+markers", name=f"{period}-year convention", connectgaps=False))
fig.update_layout(title=f"{FOCUS_CITY} · SSP585 Rx1day horizon labels",
                  xaxis_title="Published horizon", yaxis_title=f"Rx1day ({rx1day.unit})",
                  template="plotly_white")
fig.show()


## Optional historical cross-check

Set `ERA5_RX1DAY` above and rerun to compare a canonical ERA5 Rx1day baseline at `FOCUS_CITY`. The file must declare ERA5 provenance and mm/day units and contain Rx1day only. An uncovered location stops the comparison.

This compares two sources and methodologies: ERA5 annual maxima versus pooled, bias-corrected projections, with potentially different spatial resolution and temporal support. A difference cannot be attributed only to climate change. Neither curve provides an asset vulnerability or financial loss estimate.

In [8]:
if ERA5_RX1DAY is None:
    print("No ERA5 file selected. Set ERA5_RX1DAY to enable the historical cross-check.")
else:
    baseline = HazardDataset.local(Path(ERA5_RX1DAY))
    baseline_metadata = baseline.metadata()
    if (baseline_metadata.source.provider != "era5"
        or baseline_metadata.value_unit != "mm/day"
        or baseline.provider.list_hazards() != ("rx1day",)):
        raise ValueError("Comparison requires a canonical ERA5 Rx1day baseline in mm/day")
    focus_assets = pa.Table.from_pylist([
        row for row in assets.to_pylist() if row["asset_id"] == FOCUS_CITY
    ])
    historical = (baseline.for_assets(focus_assets).select(hazard_names=["rx1day"])
                  .return_periods(RETURN_PERIODS)
                  .write_parquet(OUTPUT / "era5_comparison.parquet"))
    history = pq.read_table(historical.output).to_pandas()
    if history.empty or history[list(historical.value_columns)].isna().all().all():
        raise LookupError(f"ERA5 baseline has no usable Rx1day curve for {FOCUS_CITY}")
    display(baseline_metadata.model_dump(mode="json"))
    display(history[["asset_id", "horizon", "pathway", "spatial_match", *historical.value_columns]])
    fig = go.Figure()
    for row in history.itertuples():
        fig.add_trace(go.Scatter(x=RETURN_PERIODS,
            y=[getattr(row, f"value_rp{period}") for period in RETURN_PERIODS],
            mode="lines+markers", name=f"ERA5 historical · {row.horizon}"))
    for row in focus.loc[focus.horizon.eq(HORIZON)].itertuples():
        fig.add_trace(go.Scatter(x=RETURN_PERIODS,
            y=[getattr(row, f"value_rp{period}") for period in RETURN_PERIODS],
            mode="lines+markers", name=f"CRC SSP585 · {HORIZON}"))
    fig.update_layout(title=f"{FOCUS_CITY} · Rx1day source/methodology comparison",
                      xaxis=dict(title="Return-period convention (years)", type="log"),
                      yaxis_title="Rx1day (mm/day)", template="plotly_white")
    fig.show()


No ERA5 file selected. Set ERA5_RX1DAY to enable the historical cross-check.


## Reuse the workflow

The [paired pipeline](../pipelines/crc_open_pipeline.py) writes the Toronto 2050 return values as Parquet, using the same release, SDK access path and return-period conventions. The notebook additionally writes canonical Rx1day curves and all sampled-city return values under `data/crc_open`. Preserve the catalogue, release and source metadata when sharing results.

Try another sampled city or published horizon, inspect its spatial match, and compare the outputs. For another hazard, start from the catalogue's units, tail, probability semantics and notes, then adapt the analysis accordingly. Multi-hazard materialization preserves separate scientific metadata rather than combining incompatible quantities.